In [1]:
import pathlib
from collections import OrderedDict
from typing import Any, Iterable

import ray
import numpy as np
import anndata
import pandas as pd
# from bolero.tl.dataset.sc_transforms import FilterRegions
# from bolero.tl.dataset.transforms import (
#     FetchRegionOneHot,
#     ReverseComplement,
# )

# from bolero.utils import get_global_coords, understand_regions
# from bolero.tl.dataset.file_transforms import FetchRegionBigWigs, GetEmbedding
# from utils import BorzoiRegions, clamp_sqrt_large_value

# DNA_NAME = "dna_one_hot"
from bolero.tl.model.borzoi.dataset import BorzoiDatasetOnline
from bolero import init

init(num_cpus=32, object_store_memory_gb=100)

2024-10-10 12:17:00,711	INFO worker.py:1762 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


In [2]:
data_dir = '/large_storage/zhoulab/tlgallent/data/Li2023Science/'
bw_file = f'{data_dir}/bigwig/ASC.bw'
# bed = f'{data_dir}peaks/cCREs.bed'


In [3]:
bigwig_paths = [bw_file]

In [4]:
adata = anndata.read_h5ad('/large_storage/zhoulab/tlgallent/data/cell_29000_rna_raw_gencode_adata_with_embeddings.h5ad')
scvi_embedding = adata.obsm['X_scVI']
# Create a DataFrame with embeddings and cell types
df = pd.DataFrame(scvi_embedding, index=adata.obs.index)
df['cell_type'] = adata.obs['MajorType']
# Group by cell type
grouped = df.groupby('cell_type').mean()
# Get embedding
recalculated_embedding = grouped.to_numpy()


leg_map = {item: index for index, item in enumerate(grouped.index.to_list())}
leg_map

/tmp/ipykernel_1794756/135511446.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby('cell_type').mean()


{'ASC': 0,
 'Amy': 1,
 'CHD7': 2,
 'EC': 3,
 'Foxp2': 4,
 'L4_IT': 5,
 'L5_ET': 6,
 'L5_IT': 7,
 'L6_CT': 8,
 'L6_IT': 9,
 'L6_IT_Car3': 10,
 'L6b': 11,
 'L23_IT': 12,
 'L56_NP': 13,
 'Lamp5': 14,
 'Lamp5_LHX6': 15,
 'MGC': 16,
 'MSN_D1': 17,
 'MSN_D2': 18,
 'ODC': 19,
 'OPC': 20,
 'PC': 21,
 'Pvalb': 22,
 'Pvalb_ChC': 23,
 'Sncg': 24,
 'Sst': 25,
 'SubCtx': 26,
 'VLMC': 27,
 'Vip': 28}

In [5]:
# borzoi_online_dataset = BorzoiDatasetOnline(bigwig_paths=bigwig_paths, bed=bed, leg_map=leg_map, genome="hg38")

In [6]:
# borzoi_online_dataset.train()

In [7]:
# borzoi_online_dataset.get_default_config()

In [8]:
# vars(borzoi_online_dataset)

In [9]:
# loader1 = borzoi_online_dataset.get_dataloader(folds=0, region_bed=bed)

# for batch in loader1:
#     pass

# for k, v in batch.items():
#     print(k, v.dtype, v.shape)
# del loader1

## PROCESSED DATA ##

In [10]:
borzoi_online_dataset = BorzoiDatasetOnline(bigwig_paths=bigwig_paths, bed=None, leg_map=leg_map, genome="hg38", use_borzoi_regions=True)


In [11]:
vars(borzoi_online_dataset)

{'genome': Genome: hg38
 Fasta Path: /home/tlgallent/projects/software/bolero/src/bolero/data/hg38/fasta/hg38.fa
 Genome One Hot Zarr: Not created,
 'batch_size': 2,
 'dna': False,
 '_block_size': 20,
 '_max_blocks': 200,
 'bigwig_paths': ['/large_storage/zhoulab/tlgallent/data/Li2023Science//bigwig/ASC.bw'],
 'keys': list[str],
 'leg_map': {'ASC': 0,
  'Amy': 1,
  'CHD7': 2,
  'EC': 3,
  'Foxp2': 4,
  'L4_IT': 5,
  'L5_ET': 6,
  'L5_IT': 7,
  'L6_CT': 8,
  'L6_IT': 9,
  'L6_IT_Car3': 10,
  'L6b': 11,
  'L23_IT': 12,
  'L56_NP': 13,
  'Lamp5': 14,
  'Lamp5_LHX6': 15,
  'MGC': 16,
  'MSN_D1': 17,
  'MSN_D2': 18,
  'ODC': 19,
  'OPC': 20,
  'PC': 21,
  'Pvalb': 22,
  'Pvalb_ChC': 23,
  'Sncg': 24,
  'Sst': 25,
  'SubCtx': 26,
  'VLMC': 27,
  'Vip': 28},
 'lora': bool,
 'dna_window': 524288,
 'signal_window': 524288,
 'pos_resolution': 32,
 'max_jitter': 3,
 'reverse_complement': True,
 'cov_filter_name': None,
 'clamp_sqrt_threshold': None,
 'use_borzoi_regions': True,
 'cell_type_dict':

In [12]:
(train_folds, valid_folds, test_folds, train_regions, valid_regions, test_regions) = borzoi_online_dataset.get_train_valid_test(fold=0)

In [13]:
train_regions

,Chromosome,Start,End,Name,Original_Name,fold
0,chr1,155957121,156481409,chr1:155957121-156481409,13844,2
1,chr1,152760876,153285164,chr1:152760876-153285164,13856,2
2,chr1,152121627,152645915,chr1:152121627-152645915,13912,2
3,chr1,150548091,151072379,chr1:150548091-151072379,13942,2
4,chr1,150351399,150875687,chr1:150351399-150875687,13956,2
...,...,...,...,...,...,...
39798,chrX,4650488,5174776,chrX:4650488-5174776,54865,7
39799,chrX,36285423,36809711,chrX:36285423-36809711,54879,7
39800,chrX,4847180,5371468,chrX:4847180-5371468,55180,7
39801,chrX,4355450,4879738,chrX:4355450-4879738,55197,7


In [14]:
loader1 = borzoi_online_dataset.get_dataloader(folds=0, region_bed=train_regions)

In [17]:
borzoi_online_dataset.bed

,Chromosome,Start,End,region,Original_Name,fold
0,chr1,155957121,156481409,chr1:155957121-156481409,13844,2
1,chr1,152760876,153285164,chr1:152760876-153285164,13856,2
2,chr1,152121627,152645915,chr1:152121627-152645915,13912,2
3,chr1,150548091,151072379,chr1:150548091-151072379,13942,2
4,chr1,150351399,150875687,chr1:150351399-150875687,13956,2
...,...,...,...,...,...,...
39798,chrX,4650488,5174776,chrX:4650488-5174776,54865,7
39799,chrX,36285423,36809711,chrX:36285423-36809711,54879,7
39800,chrX,4847180,5371468,chrX:4847180-5371468,55180,7
39801,chrX,4355450,4879738,chrX:4355450-4879738,55197,7


In [16]:
for batch in loader1:
    pass

print('printing batch items...')
for k, v in batch.items():
    print(k, v.dtype, v.shape)
del loader1

/home/tlgallent/miniconda/envs/bolero_dev/lib/python3.10/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
2024-10-10 12:17:19,499	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2024-10-10_12-16-58_115453_1794756/logs/ray-data
2024-10-10 12:17:19,500	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> AllToAllOperator[Repartition]


Loading genome DNA one-hot encoding...
Created remote one-hot object at ObjectRef(00ffffffffffffffffffffffffffffffffffffff0100000002e1f505)
Data loader kwargs {'batch_size': 2, 'prefetch_batches': 3, 'drop_last': True}


2024-10-10 12:18:00,270	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2024-10-10_12-16-58_115453_1794756/logs/ray-data
2024-10-10 12:18:00,271	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> ActorPoolMapOperator[MapBatches(select_columns)->MapBatches(FetchRegionBigWigsReduced)] -> ActorPoolMapOperator[MapBatches(_concat_bw_chunks)->MapBatches(FetchRegionOneHot)] -> TaskPoolMapOperator[MapBatches(_region_to_global_coords)] -> TaskPoolMapOperator[FlatMap(_convert_data)]


printing batch items...
bw_values torch.float32 torch.Size([2, 1, 16384])
dna_one_hot torch.bool torch.Size([2, 4, 524288])
Original_Name torch.int64 torch.Size([2])
region torch.int64 torch.Size([2, 2])


In [37]:
list(reversed(sorted(list(batch['bw_values'][1,0,:]))))[:15]

[tensor(4.9375),
 tensor(4.4688),
 tensor(3.8750),
 tensor(3.3125),
 tensor(3.0938),
 tensor(2.7500),
 tensor(2.5000),
 tensor(2.1875),
 tensor(2.1250),
 tensor(2.0938),
 tensor(2.0312),
 tensor(1.9375),
 tensor(1.9062),
 tensor(1.8125),
 tensor(1.5625)]

In [82]:
batch['bw_values'][1,:,:10]

tensor([[0.0000, 0.0000, 0.0000, 0.0312, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000]])

In [81]:
batch['dna_one_hot'][1,:,:322]

tensor([[False, False, False,  ...,  True,  True, False],
        [False, False, False,  ..., False, False, False],
        [ True,  True, False,  ..., False, False, False],
        [False, False,  True,  ..., False, False,  True]])

In [43]:
batch['Original_Name'][1]

tensor(36467)

In [44]:
batch['region'][1]

tensor([3022607879, 3023132167])

In [63]:
from bolero import Genome

In [64]:
genome = Genome('hg38')

In [65]:
genome.parse_global_coords(batch['region'].cpu().numpy())

,Chromosome,Start,End,Name
0,chrX,51541141,52065429,chrX:51541141-52065429
1,chrX,147606357,148130645,chrX:147606357-148130645


In [79]:
147606357 + 320

147606677

In [74]:
147911906 - 147606219

305687

## CHECK REVERSE COMPLEMENT MANUAL ##

## CHECK REGIONS MANUAL ##

## CHECK DNA ONE HOT MANUAL ##